# Slocum mission processing

Exploration notebook for taking one Slocum mission from a raw flashcard /
telemetry dump through to **L0 / L1 / L2** NetCDF.

It drives the same code as the `slocum-process-mission` CLI, but step by step
so you can inspect the data between stages. The reusable logic lives in the
`slocum_data_processing` package (`rawprep`, `processing`); this notebook only
orchestrates and plots.

| Level | What it is | Built by |
|---|---|---|
| **L0** | every decoded sample, flight + science time-merged, raw Slocum sensor names, no derived vars / no QC | `processing.pyglider_run.build_l0` (dbdreader) |
| **L1** | CF / OG1 names, TEOS-10 salinity & density, profile index, clipped to the deployment window | `pyglider.slocum.binary_to_timeseries` |
| **L2** | gridded time × depth | `pyglider.ncprocess.make_gridfiles` |

QC (auto for delayed-mode, manual for the published product) is a separate,
later step — not in this notebook.

**Prerequisites**

```bash
cd ~/projects/slocum_data_processing
python3 -m venv .venv            # if missing
.venv/bin/pip install -e "python/[notebook]"
```

- The mission data lives on the Kerberos-mounted `/Data/gfi` share — run
  `kinit` first if reads fail with *Permission denied*.
- Teledyne's `compexp` (raw-prep decompression) is only needed if `raw/`
  contains compressed `*.[dest]cd` files. It is not distributed with the
  repo; set its path in the config cell or `$SLOCUM_COMPEXP`.

See also `docs/user-guide/processing-a-mission.md`.

## 0. Setup

In [ ]:
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import dbdreader

from slocum_data_processing import rawprep
from slocum_data_processing.processing import config as mission_config
from slocum_data_processing.processing import pyglider_run
from slocum_data_processing.delayed import trigger

## 1. Mission configuration

Set the mission and the data locations. Everything else is derived.

In [ ]:
MISSION = 2                 # mission number, or a python/missions/ dir prefix ("002")

DATA_ROOT    = Path("/Data/gfi/projects/slocum/data/delayed")
MASTER_CACHE = Path("/Data/gfi/projects/slocum/data/cache")

# Teledyne compexp — only for compressed *.[dest]cd files in raw/.
# Leave None if raw/ already holds uncompressed *.dbd / *.ebd.
COMPEXP = None              # e.g. Path("~/glider/twr_tools/linux/compexp").expanduser()

# --- derived ---
_token = f"{int(MISSION):03d}" if str(MISSION).isdigit() else str(MISSION)
deployment_dir = next(d for d in sorted(DATA_ROOT.glob(f"{_token}*")) if d.is_dir())
raw_dir    = deployment_dir / "raw"
binary_dir = deployment_dir / "binary"
logs_dir   = deployment_dir / "logs"
work_dir   = deployment_dir / "pyglider"     # cache/ + L0/ + L1/ + L2/ land here

deployment_dir

## 2. Raw prep — flashcard dump → clean `binary/`

Four idempotent steps (`slocum_data_processing.rawprep`):

1. **copy** the binary + log + cache files out of `raw/` (any layout) into a
   flat `binary/`.
2. **decompress** Teledyne-compressed `*.[dest]cd` files (skipped if none).
3. **rename** 8x3 DOS names (`00490000.dbd`) → full segment names
   (`gna-2012-156-0-0.dbd`), from each file's header.
4. **sync cache** — make sure the `.cac` sensor-list caches referenced by
   compressed/factored files are available (full `.dbd`/`.ebd` carry theirs
   inline).

If `binary/` is already prepared you can skip straight to section 3.

In [ ]:
rawprep.inventory(raw_dir, binary_dir)

In [ ]:
rawprep.copy_raw_to_binary(raw_dir, binary_dir, MASTER_CACHE)   # add include_telemetry=True for *.sbd/*.tbd

In [ ]:
rawprep.decompress_dir(binary_dir, compexp=COMPEXP)             # no-op if nothing is compressed

In [ ]:
rename_report = rawprep.rename_to_full_filenames(binary_dir, logs_dir=logs_dir)
rename_report

In [ ]:
cache_report = rawprep.sync_cache_files(binary_dir, MASTER_CACHE, cache_dir=work_dir / "cache")
assert cache_report.ok, f"unresolved cache files: {cache_report.missing}"
cache_report

In [ ]:
rawprep.inventory(raw_dir, binary_dir)     # raw vs binary coverage after prep

## 3. Inspect the data before processing

Two things decide the config: **what sensors are on**, and **the real
deployment window** (old missions often bundle bench tests + a checkout dive
+ the deployment, with gaps).

In [ ]:
cache = str(work_dir / "cache")
sci = dbdreader.MultiDBD(pattern=f"{binary_dir}/*.ebd", cacheDir=cache)
science_sensors = sorted(
    p for grp in sci.parameterNames.values() for p in grp
    if p.startswith("sci_") and any(k in p for k in ("water", "flntu", "flbbcd", "oxy", "bb"))
)
science_sensors

In [ ]:
flt = dbdreader.MultiDBD(pattern=f"{binary_dir}/*.dbd", cacheDir=cache)
t, z = flt.get("m_depth")
tt = pd.to_datetime(t, unit="s")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(tt, z, ".", ms=1)
ax.invert_yaxis(); ax.set_ylabel("m_depth [m]"); ax.grid(alpha=0.3)
ax.set_title(f"{deployment_dir.name} — all flight files")
print("data span:", tt.min(), "→", tt.max())

# daily max depth — spot the deployment vs the tests
daily = pd.Series(z, index=tt).resample("D").max().dropna()
daily[daily > 5]

## 4. Mission config — `deployment.yml`

Per-mission config lives in `python/missions/<NNN-name>/` (`deployment.yml`
+ `sensors.txt`), version-controlled. To start a new mission, copy mission
002's as a template and edit the identity / payload / `processing:` block —
see `docs/user-guide/processing-a-mission.md` step 3.

Set `processing.l1_time_range` in the yaml to the window from section 3, then
reload here.

In [ ]:
cfg = mission_config.load(MISSION)
print("deployment :", cfg.deployment_name)
print("L0 window  :", cfg.l0_time_range, "(None = all data)")
print("L1 window  :", cfg.l1_time_range)
print("profiles   : filt", cfg.profile_filt_time, "s / min", cfg.profile_min_time, "s")
print("grid dz    :", cfg.grid_dz, "m")
print("sensors    :", cfg.sensor_list)
cfg.deployment["glider_devices"]

## 5. Process — L0 / L1 / L2

Run the steps individually (below) to inspect between them, or all at once:

```python
products = trigger.run_for_mission(MISSION, binary_dir, work_dir)
```

In [ ]:
work = pyglider_run.WorkDirs.under(work_dir)
work

In [ ]:
l0_path = pyglider_run.build_l0(cfg, binary_dir, work)      # all data, raw names
l0 = xr.open_dataset(l0_path)
print(dict(l0.sizes), "|", len(l0.data_vars), "sensors")
l0

In [ ]:
l1_path = pyglider_run.build_l1(cfg, binary_dir, work)      # CF names, derived S/rho, windowed
l1 = xr.open_dataset(l1_path)
l1

In [ ]:
l2_paths = pyglider_run.build_l2(cfg, work, l1_path)        # gridded time x depth
l2 = xr.open_dataset(l2_paths[0])
l2

## 6. Inspect the products

In [ ]:
def section(ds, var, ax, cmap="viridis"):
    d = ds[var].values
    ok = np.isfinite(d) & np.isfinite(ds["depth"].values)
    lo, hi = np.nanpercentile(d[ok], [1, 99])
    sc = ax.scatter(pd.to_datetime(ds["time"].values), ds["depth"].values,
                    c=d, s=3, cmap=cmap, vmin=lo, vmax=hi)
    ax.invert_yaxis(); ax.set_ylabel("depth [m]"); ax.set_title(f"L1  {var}")
    ax.figure.colorbar(sc, ax=ax, pad=0.01)

fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
for ax, v in zip(axes, ["temperature", "salinity", "potential_density"]):
    section(l1, v, ax, cmap={"salinity": "plasma", "potential_density": "cividis"}.get(v, "viridis"))
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
for ax, v in zip(axes, ["temperature", "salinity", "potential_density"]):
    l2[v].plot(ax=ax, x="time", y="depth", yincrease=False, robust=True,
               cmap={"salinity": "plasma", "potential_density": "cividis"}.get(v, "viridis"))
    ax.set_title(f"L2 gridded  {v}")
fig.tight_layout()

In [ ]:
for name, ds in [("L0", l0), ("L1", l1), ("L2", l2)]:
    print(f"{name}: {dict(ds.sizes)}  processing_level={ds.attrs.get('processing_level')}  "
          f"featureType={ds.attrs.get('featureType')}")

## 7. Next steps

- Commit the mission config: `git add python/missions/<NNN-name>/`
- Hand the L1 / L2 files to `norgliders-ERDDAP/ingest/ingest.py` — it
  registers them in OGDB and transfers them to the ERDDAP server.
- QC comes later, as its own module.